In [2]:
import os

DATASET_PATH = '/kaggle/input/competitions/aml-group-project/release'
TRAIN_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/train.csv'
IMGS_PATH = '/kaggle/input/competitions/aml-group-project/release/images'
TEST_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/test_episodes_release.csv'
SUBMISSION_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/sample_submission.csv'

In [22]:
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset

class TrainDataset(Dataset):

    def __init__(self, imgs_path: str, train_csv_path: str, transform = None):

        self.imgs_path = imgs_path
        self.df = pd.read_csv(train_csv_path)

        self.filenames = self.df['filename'].values
        self.original_labels = self.df['label'].values
        self.unique_labels = np.unique(self.original_labels)
        
        self.unique_labels.sort()

        # To create sequential labels
        self.label_to_idx = {label: idx for idx, label in enumerate(self.unique_labels)}
        self.idx_to_label = {idx: label for idx, label in enumerate(self.unique_labels)}
        
        self.mapped_labels = [self.label_to_idx[label] for label in self.original_labels]
        self.transform = transform

    def __len__(self):

        return len(self.filenames)


    def __getitem__(self, idx):
        
        fname = self.filenames[idx]
        label = self.mapped_labels[idx]
        
        img_path = os.path.join(self.imgs_path, fname)
        Im = Image.open(img_path).convert('RGB')

        if self.transform is not None:
            Im = self.transform(Im)
        
        return Im, label

In [23]:
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset
import numpy as np

class EpisodesSet(Dataset):

    def __init__(self, imgs_path: str, test_csv_path: str, transform = None):

        self.imgs_path = imgs_path
        self.df = pd.read_csv(test_csv_path)
        self.transform = transform

        self.filenames = self.df['filename'].values
        self.unique_episodes = self.df['episode_id'].unique()

    def __len__(self):

        return len(self.unique_episodes)


    def __getitem__(self, idx):

        episode_df = self.df[self.df['episode_id'] == idx]
        support_df = episode_df[episode_df['role'] == 'support']
        query_df = episode_df[episode_df['role'] == 'query']

        fnames_s = support_df['filename'].values
        support_imgs_paths = [os.path.join(self.imgs_path,f) for f in fnames_s]

        fnames_q = query_df['filename'].values
        query_imgs_paths = [os.path.join(self.imgs_path,f) for f in fnames_q]

        query_idxs = query_df['Id'].values

        support_imgs = []
        for i in range(len(support_imgs_paths)):
            img = Image.open(support_imgs_paths[i]).convert('RGB')
            if self.transform is not None:
                img = self.transform(img)
            support_imgs.append(img)

        query_imgs = []
        for i in range(len(query_imgs_paths)):
            img = Image.open(query_imgs_paths[i]).convert('RGB')
            if self.transform is not None:
                img = self.transform(img)
            query_imgs.append(img)

        raw_support_labels = np.array(support_df['label'].values).astype(int)
        unique_classes = np.unique(raw_support_labels)
        unique_classes.sort()

        label_to_local_idx = {label: i for i, label in enumerate(unique_classes)}
        support_labels = np.array([label_to_local_idx[l] for l in raw_support_labels])

        return {'support_imgs': support_imgs,
               'query_imgs': query_imgs,
               'support_labels':support_labels,
               'class_mapping':label_to_local_idx,
               'idx_submission': query_idxs}

In [24]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = TrainDataset(
    imgs_path=IMGS_PATH, 
    train_csv_path=TRAIN_CSV_PATH, 
    transform=train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,          
    shuffle=True,          
    num_workers=2,          
    pin_memory=True, 
    drop_last=True          
)

In [25]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


test_dataset = EpisodesSet(
    imgs_path=IMGS_PATH, 
    test_csv_path=TEST_CSV_PATH, 
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,           # 1 batch = 1 intero episodio (25 support + 25 query)
    shuffle=False,          
    num_workers=1,          
    pin_memory=True
)

In [40]:
for images, labels in train_loader:
    print(images.shape, labels.shape)
    break

for episode in test_loader:
    print(episode['support_imgs'][0].shape)
    break

torch.Size([64, 3, 224, 224]) torch.Size([64])
torch.Size([1, 3, 224, 224])
